In [ ]:
import pandas as pd

In [ ]:
NIFTY500_URL = "https://archives.nseindia.com/content/indices/ind_nifty500list.csv"

In [ ]:
df = pd.read_csv(NIFTY500_URL)

In [ ]:
df.head()

In [ ]:
df.Industry.value_counts()

In [ ]:
df[df["Series"] == "BE"]

In [ ]:
df[df.Symbol == "ADANIENT"]

In [ ]:
# requests.get(MARKET_DATA_URL + "ADANIENT").json()
# Not working as plain requests are blocked by NSE. We need to use a session and set the user-agent and cookies to get the data.

In [ ]:
import time
import json
import random
import requests
from pathlib import Path

In [ ]:
NSE_BASE_URL = "https://www.nseindia.com"
MARKET_DATA_URL = "https://www.nseindia.com/api/quote-equity?symbol="
REFERER_URL = "https://www.nseindia.com/get-quotes/equity?symbol="

In [ ]:
# Step 1: Create a session to persist cookies
session = requests.Session()

def get_market_data(session: requests.Session, symbol: str) -> dict:
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json",
        "Referer": REFERER_URL + symbol
    }

    # Step 2: Make an initial request to get the cookies
    session.get(NSE_BASE_URL, headers=headers)
    # Step 3: Make the request to get market data
    response = session.get(MARKET_DATA_URL + symbol, headers=headers)
    
    return response.json()

get_market_data(session, "ADANIENT")

In [ ]:
output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
def get_nifty500_metadata(session: requests.Session, symbols: list[str]) -> None:
    for symbol in symbols[:10]:
        print(f"Fetching data for {symbol}...")
        data = get_market_data(session, symbol)
        time.sleep(random.uniform(1, 7))  # Sleep to avoid hitting rate limits
        # Save the data to a file
        output_file = output_dir / f"{symbol}.json"
        with open(output_file, "w") as f:
            json.dump(data, f, indent=4)

get_nifty500_metadata(session, df.Symbol.tolist())

In [ ]:
# Should I have used playwright to fetch he data?
# Currently not needed as we are able to get the data using requests with proper headers and cookies.
# If below issues are faced, then we can consider using playwright.
# - If NSE blocks IP address after multiple requests
# - If data needs to be fetched from JS rendered pages not API endpoints

In [ ]:
BASE_HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
}

def get_headers(symbol: str, referer_url: str) -> dict[str, str]:
    return {
        **BASE_HEADERS,
        "Referer": f"{referer_url}{symbol}",
    }

In [ ]:
from typing import Optional

In [ ]:
# Retry with exonential backoff
def get_market_data_with_retry_v1(
        session: requests.Session, symbol: str, retries: int = 3
) -> Optional[dict]:
    for attempt in range(retries):
        try:
            headers = get_headers(symbol, REFERER_URL)
            session.get(NSE_BASE_URL, headers=headers)
            response = session.get(MARKET_DATA_URL + symbol, headers=headers)
            if response.status_code == 200:
                return response.json()
            elif response.status_code == 403:
                print(f"Access denied for {symbol}. Possible IP block. Attempt {attempt + 1} of {retries}.")
                session.cookies.clear()  # Clear cookies to try to bypass block
                session.get(NSE_BASE_URL, headers=headers)  # Get new cookies
            else:
                print(f"Unexpected status code {response.status_code} for {symbol}. Attempt {attempt + 1} of {retries}.")
        except Exception as e:
            print(f"Error fetching data for {symbol}: {e}")
        
        sleep_time = 2 ** attempt + random.uniform(0, 1)
        print(f"Retrying in {sleep_time:.2f} seconds...")
        time.sleep(sleep_time)
    
    raise Exception(f"Failed to fetch data for {symbol} after {retries} attempts")

get_market_data_with_retry_v1(session, "ADANIENT")

In [ ]:
import asyncio